In [7]:
import sqlite3


db_file_path = 'generated_data.db'

conn = sqlite3.connect(db_file_path)
print(f"Successfully connected to {db_file_path}")

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
for table in tables:
    print(type(table[0]))
    if(table[0] not in ["event", "payment", "session", "ticket", "station"]):
        drop_table_sql = f"DROP TABLE IF EXISTS {table[0]}"
        print(table[0])
        cursor.execute(drop_table_sql)


conn.close()

Successfully connected to generated_data.db
<class 'str'>
<class 'str'>
<class 'str'>
tariff
<class 'str'>
discount_rule
<class 'str'>
voucher
<class 'str'>
payment_discount
<class 'str'>
payment_voucher
<class 'str'>
<class 'str'>
<class 'str'>


In [23]:
from faker import Faker
import random

fake = Faker()
payments = []
payment_methods = ['cash', 'card', 'online']
num_rows = 1000
for _ in range(num_rows):
    method = random.choice(payment_methods)
        
    processor_ref = None
    if method != 'cash':
        processor_ref = f"TXN-{fake.unique.bothify(text='????####').upper()}"
        
        
    created_at_dt = fake.date_time_between(start_date='-1y', end_date='now')

    payment = (
            random.randint(1000, 9999),  # session_id
            random.randint(1, 15),       # station_id
            method,
            random.randint(100, 15000),  # amount_cents (e.g., 1.00 to 150.00)
            random.choice([0, 1]),       # approved (0=False, 1=True)
            processor_ref,
            created_at_dt.isoformat()    # created_at
    )
    payments.append(payment)

sql = ''' INSERT INTO payment(session_id, station_id, method, amount_cents, approved, processor_ref, created_at)
              VALUES(?,?,?,?,?,?,?) '''

conn = sqlite3.connect(db_file_path)

cursor = conn.cursor()
cursor.executemany(sql, payments)
conn.commit()



conn.close()

In [28]:
fake = Faker()
stations = []
station_kinds = ['entry_terminal', 'exit_terminal', 'pof']
pof_labels = ["POF-01", "POF-02"]
# "Entry Lane A", "Exit Lane A"

num_rows = 900
for _ in range(num_rows):
    kind = random.choice(station_kinds)
    if(kind == 'pof'):
        label = random.choice(pof_labels)
    elif kind == "entry_terminal":
        label = "Entry Lane A"
    else:
        label = "Exit Lane A"

    station = (
            random.randint(1, 4),       
            kind,
            label
    )
    stations.append(station)

sql = ''' INSERT INTO station(zone_id, kind, label)
              VALUES(?,?,?)'''

conn = sqlite3.connect(db_file_path)

cursor = conn.cursor()
cursor.executemany(sql, stations)
conn.commit()



conn.close()

In [ ]:
import sqlite3
import random
from faker import Faker
from datetime import datetime, timedelta

# --- Config ---
num_rows = 40000   # you can adjust this
fake = Faker()
db_file_path = 'generated_data.db'

session_statuses = ['active', 'paid', 'exited', 'closed', 'overdue']
status_weights = [0.15, 0.05, 0.50, 0.25, 0.05]

entry_station_ids = [1, 2, 3, 4]
exit_station_ids = [5, 6]

def generate_plate():
    """Generates a random Moldovan-style license plate."""
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    return f"{random.choice(letters)}{random.choice(letters)}{random.choice(letters)}{random.randint(100, 999)}"

def random_date_aug_sep(year: int):
    """Pick a random datetime between 15 Aug – 15 Sep of a given year."""
    start = datetime(year, 8, 15, 0, 0, 0)
    end = datetime(year, 9, 15, 23, 59, 59)
    delta = end - start
    random_sec = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_sec)

sessions = []
current_ticket_id = 10000 

print(f"Generating {num_rows} fake session records...")

for i in range(num_rows):
    ticket_id = None
    exit_time = None
    exit_station = None
    amount_due_cents = 0
    amount_paid_cents = 0
    paid_until = None
    licence_plate_exit = None

    # --- Base Information ---
    status = random.choices(session_statuses, weights=status_weights, k=1)[0]

    year = random.choice([2022, 2023, 2024, 2025])
    entry_time = random_date_aug_sep(year)
    licence_plate_entry = generate_plate()
    entry_station = random.choice(entry_station_ids)

    # --- Introduce occasional OUTLIERS ---
    outlier = random.random() < 0.03  # ~3% chance

    if status in ['exited', 'closed']:
        duration_minutes = random.randint(15, 720)  # normal stay
        if outlier:
            duration_minutes *= random.randint(5, 20)  # extreme long stay
        duration = timedelta(minutes=duration_minutes)

        exit_time = entry_time + duration
        exit_station = random.choice(exit_station_ids)

        amount_due_cents = int(duration.total_seconds() / 60) * 10
        if outlier:
            amount_due_cents *= random.randint(2, 5)  # extreme charges
        amount_paid_cents = amount_due_cents

        paid_until = exit_time + timedelta(minutes=15)
        licence_plate_exit = licence_plate_entry if random.random() > 0.05 else generate_plate()

    elif status == 'paid':
        duration_so_far = datetime.now() - entry_time
        amount_due_cents = int(duration_so_far.total_seconds() / 60) * 10
        amount_paid_cents = amount_due_cents
        paid_until = datetime.now() + timedelta(hours=1)

    elif status == 'active':
        duration_so_far = datetime.now() - entry_time
        amount_due_cents = int(duration_so_far.total_seconds() / 60) * 10
        if outlier:
            amount_due_cents *= random.randint(2, 5)

    elif status == 'overdue':
        entry_time = random_date_aug_sep(year) - timedelta(days=random.randint(2, 10))
        duration_so_far = datetime.now() - entry_time
        amount_due_cents = int(duration_so_far.total_seconds() / 60) * 10
        amount_paid_cents = 0

    # --- Assign ticket ID ---
    if random.random() < 0.8:
        ticket_id = current_ticket_id
        current_ticket_id += 1

    session_tuple = (
        ticket_id,
        entry_time.isoformat(),
        entry_station,
        exit_time.isoformat() if exit_time else None,
        exit_station,
        status,
        amount_due_cents,
        amount_paid_cents,
        paid_until.isoformat() if paid_until else None,
        licence_plate_entry,
        licence_plate_exit
    )
    sessions.append(session_tuple)

# --- Database Insertion ---
sql = ''' INSERT INTO session(
            ticket_id, entry_time, entry_station, exit_time, exit_station,
            status, amount_due_cents, amount_paid_cents, paid_until,
            licence_plate_entry, licence_plate_exit
          ) VALUES(?,?,?,?,?,?,?,?,?,?,?) '''

try:
    conn = sqlite3.connect(db_file_path)
    cursor = conn.cursor()
    
    cursor.executemany(sql, sessions)
    conn.commit()
    
    print(f"✅ Successfully inserted {cursor.rowcount} rows into the 'session' table.")

except sqlite3.Error as error:
    print(f"❌ Failed to insert data into sqlite table: {error}")

finally:
    if conn:
        conn.close()
        print("🔒 The SQLite connection is closed.")

Generating 40000 fake session records...
✅ Successfully inserted 40000 rows into the 'session' table.
🔒 The SQLite connection is closed.
